## Prepare dataset

In [5]:
# ! pip install datasets

In [4]:
from datasets import load_dataset

In [6]:
# Load the IMDB dataset
dataset = load_dataset("imdb")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

unsupervised-00000-of-00001.parquet:   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [7]:
# Split into train and validation sets
train_dataset = dataset["train"]
test_dataset = dataset["test"]

In [8]:
print(train_dataset)
print(test_dataset)

Dataset({
    features: ['text', 'label'],
    num_rows: 25000
})
Dataset({
    features: ['text', 'label'],
    num_rows: 25000
})


 ## Set up the tokenizer

In [9]:
from transformers import AutoTokenizer

In [10]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [11]:
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

## Create DataLoaders

In [12]:
import torch
from torch.utils.data import DataLoader

In [13]:
# Prepare the datasets
train_dataset = tokenized_train.remove_columns(["text"])
train_dataset = train_dataset.rename_column("label", "labels")
train_dataset.set_format("torch")

test_dataset = tokenized_test.remove_columns(["text"])
test_dataset = test_dataset.rename_column("label", "labels")
test_dataset.set_format("torch")

# Create DataLoaders
batch_size = 16
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size)

## Implement the RNN model

In [14]:
import torch.nn as nn

class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim, n_layers,
                 bidirectional, dropout, pad_idx):
        super().__init__()

        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)

        # RNN layer (can use GRU or LSTM)
        self.rnn = nn.LSTM(embed_dim,
                          hidden_dim,
                          num_layers=n_layers,
                          bidirectional=bidirectional,
                          dropout=dropout,
                          batch_first=True)

        # Fully connected layer
        self.fc = nn.Linear(hidden_dim * 2 if bidirectional else hidden_dim, output_dim)

        # Dropout layer
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids, attention_mask=None):
        # input_ids: [batch size, sequence length]

        # embedded: [batch size, sequence length, embedding dim]
        embedded = self.embedding(input_ids)

        # Apply RNN
        # output: [batch size, sequence length, hidden dim * num directions]
        # hidden: [num layers * num directions, batch size, hidden dim]
        output, (hidden, cell) = self.rnn(embedded)

        # If bidirectional, concatenate the final forward and backward hidden states
        if self.rnn.bidirectional:
            hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        else:
            hidden = hidden[-1,:,:]

        # Apply dropout
        hidden = self.dropout(hidden)

        # Pass through the fully-connected layer
        # output: [batch size, output dim]
        return self.fc(hidden)

##  Initialize the model

In [15]:
# Model parameters
vocab_size = tokenizer.vocab_size
embed_dim = 300
hidden_dim = 256
output_dim = 2  # Binary classification
n_layers = 2
bidirectional = True
dropout = 0.25
pad_idx = tokenizer.pad_token_id

# Create the model
model = SentimentRNN(vocab_size, embed_dim, hidden_dim, output_dim,
                     n_layers, bidirectional, dropout, pad_idx)

# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Define optimizer and loss function
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

## Training loop

In [16]:
def train(model, dataloader, optimizer, criterion, device):
    model.train()

    epoch_loss = 0
    epoch_acc = 0

    for batch in dataloader:
        # Move data to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Forward pass
        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)

        # Calculate loss and accuracy
        loss = criterion(outputs, labels)
        preds = torch.argmax(outputs, dim=1)
        correct = (preds == labels).float().sum()
        acc = correct / len(labels)

        # Backward pass
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        epoch_acc += acc.item()

    return epoch_loss / len(dataloader), epoch_acc / len(dataloader)

## Evaluation function

In [17]:
def evaluate(model, dataloader, criterion, device):
    model.eval()

    epoch_loss = 0
    epoch_acc = 0

    with torch.no_grad():
        for batch in dataloader:
            # Move data to device
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # Forward pass
            outputs = model(input_ids, attention_mask)

            # Calculate loss and accuracy
            loss = criterion(outputs, labels)
            preds = torch.argmax(outputs, dim=1)
            correct = (preds == labels).float().sum()
            acc = correct / len(labels)

            epoch_loss += loss.item()
            epoch_acc += acc.item()

    return epoch_loss / len(dataloader), epoch_acc / len(dataloader)

## Train the model

In [18]:
num_epochs = 5

for epoch in range(num_epochs):
    train_loss, train_acc = train(model, train_dataloader, optimizer, criterion, device)
    val_loss, val_acc = evaluate(model, test_dataloader, criterion, device)

    print(f'Epoch: {epoch+1}')
    print(f'\tTrain Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}%')
    print(f'\tVal. Loss: {val_loss:.3f} | Val. Acc: {val_acc*100:.2f}%')

Epoch: 1
	Train Loss: 0.688 | Train Acc: 54.12%
	Val. Loss: 0.674 | Val. Acc: 57.80%
Epoch: 2
	Train Loss: 0.586 | Train Acc: 69.62%
	Val. Loss: 0.529 | Val. Acc: 74.43%
Epoch: 3
	Train Loss: 0.477 | Train Acc: 77.72%
	Val. Loss: 0.471 | Val. Acc: 79.22%
Epoch: 4
	Train Loss: 0.424 | Train Acc: 80.95%
	Val. Loss: 0.483 | Val. Acc: 78.92%
Epoch: 5
	Train Loss: 0.387 | Train Acc: 83.39%
	Val. Loss: 0.423 | Val. Acc: 81.57%


In [23]:
def predict_sentiment(model, tokenizer, sentence, device):
    model.eval()

    # Tokenize the sentence
    tokens = tokenizer(sentence, padding="max_length", truncation=True, max_length=256, return_tensors="pt")

    input_ids = tokens["input_ids"].to(device)
    attention_mask = tokens["attention_mask"].to(device)

    # Make prediction
    with torch.no_grad():
        output = model(input_ids, attention_mask)

    # Get prediction
    prediction = torch.argmax(output, dim=1).item()
    probability = torch.softmax(output, dim=1)[0][prediction].item()

    return prediction, probability

# Example usage
sentence = "The movie was a complete waste of time. The plot was boring, and the acting was terrible. I would not recommend it to anyone."
prediction, probability = predict_sentiment(model, tokenizer, sentence, device)
sentiment = "positive" if prediction == 1 else "negative"
print(f"Sentiment: {sentiment} (confidence: {probability:.2f})")

Sentiment: negative (confidence: 0.93)
